In [19]:
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kruskal
from scikit_posthocs import posthoc_dunn
import warnings
warnings.filterwarnings('ignore')

# Настройка стиля графиков
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10


In [20]:
# Определяем типы швов для анализа
suture_types = ['COOS', 'ILS', 'IOVS_external', 'IOVS_internal']

def load_suture_data(suture_type):
    """Загружаем данные для конкретного типа шва и извлекаем F1-score"""
    pkl_path = f'code and results/{suture_type}/architecture_results_cv.pkl'
    
    try:
        with open(pkl_path, 'rb') as f:
            data = pickle.load(f)
        
        # Формируем выборки F1-score для каждой архитектуры
        architecture_f1_scores = {}
        
        for architecture, folds in data.items():
            f1_scores = []
            for fold_data in folds:
                f1_score = fold_data['f1']  # Извлекаем F1-score напрямую
                f1_scores.append(f1_score)
            architecture_f1_scores[architecture] = f1_scores
        
        return architecture_f1_scores
    
    except FileNotFoundError:
        print(f"⚠️ Файл {pkl_path} не найден. Пропускаем {suture_type}")
        return None

# Загружаем данные для всех типов швов
all_data = {}
for suture_type in suture_types:
    data = load_suture_data(suture_type)
    if data is not None:
        all_data[suture_type] = data
        print(f"✅ Загружены F1-scores для {suture_type}: {len(data)} архитектур")

print(f"\n📊 Всего загружено данных для {len(all_data)} типов швов")
print(f"🎯 Анализируемая метрика: F1-score (гармоническое среднее precision и recall)")


✅ Загружены F1-scores для COOS: 8 архитектур
✅ Загружены F1-scores для ILS: 8 архитектур
✅ Загружены F1-scores для IOVS_external: 8 архитектур
✅ Загружены F1-scores для IOVS_internal: 8 архитектур

📊 Всего загружено данных для 4 типов швов
🎯 Анализируемая метрика: F1-score (гармоническое среднее precision и recall)


In [21]:
# Проверяем структуру данных
for suture_type, architectures in all_data.items():
    print(f"\n=== {suture_type} ===")
    print(f"Архитектуры: {list(architectures.keys())}")
    print(f"Количество фолдов для каждой архитектуры:")
    for arch, f1_scores in architectures.items():
        print(f"  {arch}: {len(f1_scores)} фолдов")
        print(f"    F1-scores: {[round(f1, 4) for f1 in f1_scores]}")
        print(f"    Медиана F1: {np.median(f1_scores):.4f}, Размах: {np.max(f1_scores) - np.min(f1_scores):.4f}")



=== COOS ===
Архитектуры: ['EfficientNetB0', 'ResNet50V2', 'MobileNetV3Large', 'VGG16', 'VGG19', 'DenseNet121', 'InceptionV3', 'Xception']
Количество фолдов для каждой архитектуры:
  EfficientNetB0: 5 фолдов
    F1-scores: [np.float64(0.8333), np.float64(0.7143), np.float64(0.7273), np.float64(0.7273), np.float64(0.6667)]
    Медиана F1: 0.7273, Размах: 0.1667
  ResNet50V2: 5 фолдов
    F1-scores: [np.float64(0.8333), np.float64(0.7143), np.float64(0.7273), np.float64(0.8), np.float64(0.5)]
    Медиана F1: 0.7273, Размах: 0.3333
  MobileNetV3Large: 5 фолдов
    F1-scores: [np.float64(0.7273), np.float64(0.7273), np.float64(0.8), np.float64(0.6667), np.float64(0.7273)]
    Медиана F1: 0.7273, Размах: 0.1333
  VGG16: 5 фолдов
    F1-scores: [np.float64(0.6), np.float64(0.7143), np.float64(0.8), np.float64(0.8), np.float64(0.6667)]
    Медиана F1: 0.7143, Размах: 0.2000
  VGG19: 5 фолдов
    F1-scores: [np.float64(0.75), np.float64(0.7143), np.float64(0.6154), np.float64(0.8), np.float64

In [22]:
# Создаем описательную статистику для каждого типа шва
descriptive_stats = {}

for suture_type, architectures in all_data.items():
    print(f"\n{'='*60}")
    print(f"ОПИСАТЕЛЬНАЯ СТАТИСТИКА F1-SCORE ДЛЯ {suture_type}")
    print(f"{'='*60}")
    
    stats_data = []
    
    for arch, f1_scores in architectures.items():
        stats_dict = {
            'Архитектура': arch,
            'Медиана F1': np.median(f1_scores),
            'Среднее F1': np.mean(f1_scores),
            'СКО F1': np.std(f1_scores, ddof=1),
            'Мин F1': np.min(f1_scores),
            'Макс F1': np.max(f1_scores),
            'IQR F1': np.percentile(f1_scores, 75) - np.percentile(f1_scores, 25),
            'CV F1 (%)': (np.std(f1_scores, ddof=1) / np.mean(f1_scores)) * 100  # Коэффициент вариации
        }
        stats_data.append(stats_dict)
    
    # Создаем DataFrame и сортируем по медиане F1-score
    stats_df = pd.DataFrame(stats_data)
    stats_df = stats_df.sort_values('Медиана F1', ascending=False)
    
    print("\n📊 СТАТИСТИКИ F1-SCORE (отсортировано по медиане):")
    print(stats_df.round(4).to_string(index=False))
    
    # Сохраняем для дальнейшего анализа
    descriptive_stats[suture_type] = stats_df
    
    # Дополнительный анализ
    median_range = stats_df['Медиана F1'].max() - stats_df['Медиана F1'].min()
    mean_cv = stats_df['CV F1 (%)'].mean()
    
    print(f"\n🔢 Размах медиан F1: {stats_df['Медиана F1'].max():.4f} - {stats_df['Медиана F1'].min():.4f} = {median_range:.4f}")
    print(f"📈 Лучшая архитектура (по медиане F1): {stats_df.iloc[0]['Архитектура']} ({stats_df.iloc[0]['Медиана F1']:.4f})")
    print(f"📉 Худшая архитектура (по медиане F1): {stats_df.iloc[-1]['Архитектура']} ({stats_df.iloc[-1]['Медиана F1']:.4f})")
    print(f"📊 Средний коэффициент вариации: {mean_cv:.2f}% ({'Низкая' if mean_cv < 10 else 'Умеренная' if mean_cv < 20 else 'Высокая'} изменчивость)")
    
    # Классификация производительности
    high_performers = stats_df[stats_df['Медиана F1'] >= 0.8]
    medium_performers = stats_df[(stats_df['Медиана F1'] >= 0.6) & (stats_df['Медиана F1'] < 0.8)]
    low_performers = stats_df[stats_df['Медиана F1'] < 0.6]
    
    print(f"\n🎯 КЛАССИФИКАЦИЯ АРХИТЕКТУР ПО F1-SCORE:")
    print(f"   🟢 Высокая производительность (F1 ≥ 0.8): {len(high_performers)} архитектур")
    if len(high_performers) > 0:
        print(f"      {', '.join(high_performers['Архитектура'].tolist())}")
    print(f"   🟡 Средняя производительность (0.6 ≤ F1 < 0.8): {len(medium_performers)} архитектур")
    if len(medium_performers) > 0:
        print(f"      {', '.join(medium_performers['Архитектура'].tolist())}")
    print(f"   🔴 Низкая производительность (F1 < 0.6): {len(low_performers)} архитектур")
    if len(low_performers) > 0:
        print(f"      {', '.join(low_performers['Архитектура'].tolist())}")



ОПИСАТЕЛЬНАЯ СТАТИСТИКА F1-SCORE ДЛЯ COOS

📊 СТАТИСТИКИ F1-SCORE (отсортировано по медиане):
     Архитектура  Медиана F1  Среднее F1  СКО F1  Мин F1  Макс F1  IQR F1  CV F1 (%)
     DenseNet121      0.8333      0.7881  0.0791  0.6667   0.8571  0.0833    10.0428
        Xception      0.8333      0.7402  0.1709  0.5000   0.9091  0.2083    23.0910
  EfficientNetB0      0.7273      0.7338  0.0610  0.6667   0.8333  0.0130     8.3121
      ResNet50V2      0.7273      0.7150  0.1300  0.5000   0.8333  0.0857    18.1829
MobileNetV3Large      0.7273      0.7297  0.0473  0.6667   0.8000  0.0000     6.4763
           VGG16      0.7143      0.7162  0.0866  0.6000   0.8000  0.1333    12.0930
           VGG19      0.7143      0.6850  0.1032  0.5455   0.8000  0.1346    15.0697
     InceptionV3      0.7059      0.6762  0.1208  0.5263   0.8333  0.1390    17.8663

🔢 Размах медиан F1: 0.8333 - 0.7059 = 0.1275
📈 Лучшая архитектура (по медиане F1): DenseNet121 (0.8333)
📉 Худшая архитектура (по медиане F1)

In [23]:
# Проводим тест Краскела-Уоллиса для каждого типа шва
kruskal_results = {}

for suture_type, architectures in all_data.items():
    print(f"\n{'='*60}")
    print(f"ТЕСТ КРАСКЕЛА-УОЛЛИСА ДЛЯ {suture_type} (F1-SCORE)")
    print(f"{'='*60}")
    
    # Подготавливаем данные для теста
    f1_scores_list = list(architectures.values())
    architecture_names = list(architectures.keys())
    
    print(f"🔬 Сравниваем F1-score {len(architecture_names)} архитектур:")
    for i, (arch, f1_scores) in enumerate(architectures.items()):
        median_f1 = np.median(f1_scores)
        mean_f1 = np.mean(f1_scores)
        std_f1 = np.std(f1_scores, ddof=1)
        print(f"   {i+1}. {arch}: медиана F1 = {median_f1:.4f}, среднее F1 = {mean_f1:.4f} ± {std_f1:.4f}")
    
    # Проводим тест Краскела-Уоллиса
    h_statistic, p_value = kruskal(*f1_scores_list)
    
    # Вычисляем дополнительные статистики
    total_n = sum(len(f1_scores) for f1_scores in f1_scores_list)
    degrees_freedom = len(f1_scores_list) - 1
    
    # Эффект размера (приблизительная оценка)
    # Используем формулу для eta-squared для Kruskal-Wallis
    eta_squared = (h_statistic - degrees_freedom) / (total_n - degrees_freedom - 1)
    eta_squared = max(0, eta_squared)  # Не может быть отрицательным
    
    effect_size_interpretation = "Малый" if eta_squared < 0.01 else "Средний" if eta_squared < 0.06 else "Большой"
    
    # Выводим результаты
    print(f"\n📊 РЕЗУЛЬТАТЫ ТЕСТА КРАСКЕЛА-УОЛЛИСА:")
    print(f"   H-статистика: {h_statistic:.4f}")
    print(f"   Степени свободы: {degrees_freedom}")
    print(f"   p-value: {p_value:.6f}")
    print(f"   Общий размер выборки: {total_n}")
    print(f"   Эффект размера (η²): {eta_squared:.4f} ({effect_size_interpretation})")
    
    # Интерпретация результатов
    is_significant = p_value < 0.05
    
    if is_significant:
        print(f"\n   ✅ ЗНАЧИМЫЕ РАЗЛИЧИЯ (p < 0.05)")
        print(f"   🎯 Вывод: Есть статистически значимые различия между медианами F1-score")
        print(f"   📈 Интерпретация: Архитектуры показывают различную эффективность классификации")
        print(f"   🎪 Практическое значение: {effect_size_interpretation.lower()} размер эффекта")
    else:
        print(f"\n   ❌ НЕТ ЗНАЧИМЫХ РАЗЛИЧИЙ (p ≥ 0.05)")
        print(f"   🎯 Вывод: Нет статистически значимых различий между медианами F1-score")
        print(f"   📊 Интерпретация: Архитектуры показывают сопоставимую эффективность классификации")
        print(f"   ⚖️ Практическое значение: Выбор архитектуры может основываться на других критериях")
    
    # Сохраняем результаты
    kruskal_results[suture_type] = {
        'h_statistic': h_statistic,
        'p_value': p_value,
        'degrees_freedom': degrees_freedom,
        'significant': is_significant,
        'total_n': total_n,
        'architectures': architecture_names,
        'eta_squared': eta_squared,
        'effect_size': effect_size_interpretation
    }



ТЕСТ КРАСКЕЛА-УОЛЛИСА ДЛЯ COOS (F1-SCORE)
🔬 Сравниваем F1-score 8 архитектур:
   1. EfficientNetB0: медиана F1 = 0.7273, среднее F1 = 0.7338 ± 0.0610
   2. ResNet50V2: медиана F1 = 0.7273, среднее F1 = 0.7150 ± 0.1300
   3. MobileNetV3Large: медиана F1 = 0.7273, среднее F1 = 0.7297 ± 0.0473
   4. VGG16: медиана F1 = 0.7143, среднее F1 = 0.7162 ± 0.0866
   5. VGG19: медиана F1 = 0.7143, среднее F1 = 0.6850 ± 0.1032
   6. DenseNet121: медиана F1 = 0.8333, среднее F1 = 0.7881 ± 0.0791
   7. InceptionV3: медиана F1 = 0.7059, среднее F1 = 0.6762 ± 0.1208
   8. Xception: медиана F1 = 0.8333, среднее F1 = 0.7402 ± 0.1709

📊 РЕЗУЛЬТАТЫ ТЕСТА КРАСКЕЛА-УОЛЛИСА:
   H-статистика: 4.9452
   Степени свободы: 7
   p-value: 0.666656
   Общий размер выборки: 40
   Эффект размера (η²): 0.0000 (Малый)

   ❌ НЕТ ЗНАЧИМЫХ РАЗЛИЧИЙ (p ≥ 0.05)
   🎯 Вывод: Нет статистически значимых различий между медианами F1-score
   📊 Интерпретация: Архитектуры показывают сопоставимую эффективность классификации
   ⚖️ Пра

In [24]:
# Проводим post-hoc анализ с помощью Dunn's test
dunn_results = {}

for suture_type, architectures in all_data.items():
    print(f"\n{'='*60}")
    print(f"DUNN'S TEST ДЛЯ {suture_type} (F1-SCORE)")
    print(f"{'='*60}")
    
    kruskal_result = kruskal_results[suture_type]
    
    if not kruskal_result['significant']:
        print(f"⚠️ Post-hoc анализ не требуется - тест Краскела-Уоллиса не показал значимых различий в F1-score")
        dunn_results[suture_type] = None
        continue
    
    print(f"🔍 Проводим попарное сравнение F1-score с помощью Dunn's test:")
    print(f"   Поправка на множественные сравнения: Бонферрони")
    print(f"   Цель: Выявить пары архитектур с значимо различающейся эффективностью классификации")
    
    # Подготавливаем данные для Dunn теста
    data_for_dunn = []
    groups_for_dunn = []
    
    for arch, f1_scores in architectures.items():
        data_for_dunn.extend(f1_scores)
        groups_for_dunn.extend([arch] * len(f1_scores))
    
    df_for_dunn = pd.DataFrame({
        'f1_score': data_for_dunn,
        'architecture': groups_for_dunn
    })
    
    # Dunn test с поправкой Бонферрони
    dunn_matrix = posthoc_dunn(df_for_dunn, val_col='f1_score', group_col='architecture', p_adjust='bonferroni')
    
    print(f"\n📊 Матрица p-values (с поправкой Бонферрони):")
    print(dunn_matrix.round(6))
    
    # Парсим значимые результаты
    significant_pairs = []
    all_pairs = []
    architectures_list = list(architectures.keys())
    
    for i, arch1 in enumerate(architectures_list):
        for j, arch2 in enumerate(architectures_list):
            if i < j:  # Избегаем дублирования
                p_value = dunn_matrix.loc[arch1, arch2]
                if not pd.isna(p_value):
                    is_significant = p_value < 0.05
                    pair_info = {
                        'group1': arch1,
                        'group2': arch2,
                        'p_adj': p_value,
                        'significant': is_significant
                    }
                    all_pairs.append(pair_info)
                    
                    if is_significant:
                        # Получаем медианы F1-score для сравнения
                        median1_f1 = np.median(architectures[arch1])
                        median2_f1 = np.median(architectures[arch2])
                        mean1_f1 = np.mean(architectures[arch1])
                        mean2_f1 = np.mean(architectures[arch2])
                        significant_pairs.append((arch1, arch2, p_value, median1_f1, median2_f1, mean1_f1, mean2_f1))
    
    # Выводим результаты
    total_comparisons = len(all_pairs)
    significant_count = len(significant_pairs)
    
    print(f"\n🎯 РЕЗУЛЬТАТЫ ПОПАРНЫХ СРАВНЕНИЙ F1-SCORE:")
    print(f"   Всего сравнений: {total_comparisons}")
    print(f"   Значимых различий: {significant_count}")
    print(f"   Процент значимых: {significant_count/total_comparisons*100:.1f}%")
    
    if significant_pairs:
        print(f"\n   ✅ ЗНАЧИМЫЕ РАЗЛИЧИЯ В F1-SCORE (p < 0.05):")
        # Сортируем по p-value
        significant_pairs.sort(key=lambda x: x[2])
        
        for arch1, arch2, p_val, med1, med2, mean1, mean2 in significant_pairs:
            better = arch1 if med1 > med2 else arch2
            worse = arch2 if med1 > med2 else arch1
            diff_median = abs(med1 - med2)
            diff_mean = abs(mean1 - mean2)
            
            better_median = max(med1, med2)
            worse_median = min(med1, med2)
            
            print(f"      🏆 {better} > {worse}:")
            print(f"         p-value: {p_val:.6f}")
            print(f"         Δ медиана F1: {diff_median:.4f} ({better_median:.4f} vs {worse_median:.4f})")
            print(f"         Δ среднее F1: {diff_mean:.4f}")
            
            # Интерпретация практической значимости
            if diff_median >= 0.1:
                practical_significance = "Большая практическая значимость"
            elif diff_median >= 0.05:
                practical_significance = "Средняя практическая значимость"
            else:
                practical_significance = "Малая практическая значимость"
            
            print(f"         Практическая значимость: {practical_significance}")
            print()
    else:
        print(f"\n   ❌ Нет значимых попарных различий в F1-score после поправки Бонферрони")
        print(f"   💡 Возможные причины:")
        print(f"      • Консервативность поправки Бонферрони")
        print(f"      • Малый размер выборки (n=5)")
        print(f"      • Действительно схожая производительность архитектур")
    
    # Дополнительный анализ: топ и худшие архитектуры
    desc_stats = descriptive_stats[suture_type]
    print(f"\n📊 РЕЙТИНГ АРХИТЕКТУР ПО МЕДИАНЕ F1-SCORE:")
    for i, row in desc_stats.head(3).iterrows():
        print(f"   🥇 Топ-{desc_stats.index.get_loc(i)+1}: {row['Архитектура']} (медиана F1: {row['Медиана F1']:.4f})")
    
    # Сохраняем результаты
    dunn_results[suture_type] = {
        'matrix': dunn_matrix,
        'significant_pairs': significant_pairs,
        'all_pairs': all_pairs,
        'total_comparisons': total_comparisons,
        'significant_count': significant_count
    }
    
    print(f"\n" + "-"*60)



DUNN'S TEST ДЛЯ COOS (F1-SCORE)
⚠️ Post-hoc анализ не требуется - тест Краскела-Уоллиса не показал значимых различий в F1-score

DUNN'S TEST ДЛЯ ILS (F1-SCORE)
⚠️ Post-hoc анализ не требуется - тест Краскела-Уоллиса не показал значимых различий в F1-score

DUNN'S TEST ДЛЯ IOVS_external (F1-SCORE)
⚠️ Post-hoc анализ не требуется - тест Краскела-Уоллиса не показал значимых различий в F1-score

DUNN'S TEST ДЛЯ IOVS_internal (F1-SCORE)
⚠️ Post-hoc анализ не требуется - тест Краскела-Уоллиса не показал значимых различий в F1-score


In [25]:
# Создаем итоговую сводку статистического анализа F1-score
print(f"\n{'='*80}")
print(f"ИТОГОВАЯ СВОДКА СТАТИСТИЧЕСКОГО АНАЛИЗА F1-SCORE")
print(f"{'='*80}")

summary_data = []

for suture_type in all_data.keys():
    kruskal_result = kruskal_results[suture_type]
    dunn_result = dunn_results[suture_type]
    
    # Считаем количество значимых попарных различий
    significant_pairs_count = 0
    total_pairs_count = 0
    
    if dunn_result is not None:
        significant_pairs_count = dunn_result['significant_count']
        total_pairs_count = dunn_result['total_comparisons']
    
    # Получаем лучшую и худшую архитектуры по медиане F1
    desc_stats = descriptive_stats[suture_type]
    best_arch = desc_stats.iloc[0]['Архитектура']
    worst_arch = desc_stats.iloc[-1]['Архитектура']
    best_median = desc_stats.iloc[0]['Медиана F1']
    worst_median = desc_stats.iloc[-1]['Медиана F1']
    
    summary_data.append({
        'Тип шва': suture_type,
        'Архитектуры': len(all_data[suture_type]),
        'H-статистика': f"{kruskal_result['h_statistic']:.4f}",
        'p-value (Kruskal-Wallis)': f"{kruskal_result['p_value']:.6f}",
        'Эффект (η²)': f"{kruskal_result['eta_squared']:.4f}",
        'Значимость': 'Да' if kruskal_result['significant'] else 'Нет',
        'Значимые пары': f"{significant_pairs_count}/{total_pairs_count}" if total_pairs_count > 0 else "0/0",
        'Лучшая архитектура (F1)': f"{best_arch} ({best_median:.4f})",
        'Худшая архитектура (F1)': f"{worst_arch} ({worst_median:.4f})",
        'Размах медиан F1': f"{best_median - worst_median:.4f}"
    })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

# Общие выводы
print(f"\n{'='*80}")
print(f"ОБЩИЕ ВЫВОДЫ ПО F1-SCORE")
print(f"{'='*80}")

total_significant = sum(1 for result in kruskal_results.values() if result['significant'])
total_sutures = len(kruskal_results)
total_significant_pairs = sum(dunn_results[st]['significant_count'] for st in all_data.keys() if dunn_results[st] is not None)
total_all_pairs = sum(dunn_results[st]['total_comparisons'] for st in all_data.keys() if dunn_results[st] is not None)

# Анализ размеров эффекта
large_effects = sum(1 for result in kruskal_results.values() if result['eta_squared'] >= 0.06)
medium_effects = sum(1 for result in kruskal_results.values() if 0.01 <= result['eta_squared'] < 0.06)
small_effects = sum(1 for result in kruskal_results.values() if result['eta_squared'] < 0.01)

print(f"📊 Проанализировано типов швов: {total_sutures}")
print(f"🔬 Анализируемая метрика: F1-score (гармоническое среднее precision и recall)")
print(f"🧪 Использован тест: Краскела-Уоллиса (непараметрический)")
print(f"✅ Типов с значимыми различиями: {total_significant}/{total_sutures} ({total_significant/total_sutures*100:.1f}%)")
print(f"❌ Типов без значимых различий: {total_sutures - total_significant}/{total_sutures} ({(total_sutures - total_significant)/total_sutures*100:.1f}%)")
print(f"🎯 Всего значимых попарных сравнений: {total_significant_pairs}/{total_all_pairs} ({total_significant_pairs/total_all_pairs*100:.1f}%)" if total_all_pairs > 0 else "🎯 Попарные сравнения не проводились")

print(f"\n📏 РАЗМЕРЫ ЭФФЕКТА (η²):")
print(f"   🔴 Большой эффект (η² ≥ 0.06): {large_effects}/{total_sutures} ({large_effects/total_sutures*100:.1f}%)")
print(f"   🟡 Средний эффект (0.01 ≤ η² < 0.06): {medium_effects}/{total_sutures} ({medium_effects/total_sutures*100:.1f}%)")
print(f"   🟢 Малый эффект (η² < 0.01): {small_effects}/{total_sutures} ({small_effects/total_sutures*100:.1f}%)")

print(f"\n🎯 РЕКОМЕНДАЦИИ ПО ВЫБОРУ АРХИТЕКТУРЫ:")
for suture_type in all_data.keys():
    kruskal_result = kruskal_results[suture_type]
    dunn_result = dunn_results[suture_type]
    desc_stats = descriptive_stats[suture_type]
    best_arch = desc_stats.iloc[0]['Архитектура']
    best_f1 = desc_stats.iloc[0]['Медиана F1']
    effect_size = kruskal_result['effect_size']
    
    if kruskal_result['significant']:
        if dunn_result and dunn_result['significant_count'] > 0:
            print(f"   ✅ {suture_type}: Различия в F1-score статистически значимы ({dunn_result['significant_count']} пар, {effect_size.lower()} эффект)")
            print(f"      🏆 Рекомендуемая архитектура: {best_arch} (медиана F1: {best_f1:.4f})")
        else:
            print(f"   ⚠️ {suture_type}: Общие различия есть ({effect_size.lower()} эффект), но нет значимых попарных различий")
            print(f"      💡 Рассмотрите: {best_arch} (медиана F1: {best_f1:.4f})")
    else:
        print(f"   ➡️ {suture_type}: F1-score архитектур статистически неразличим ({effect_size.lower()} эффект)")
        print(f"      🎯 Выбор может основываться на других критериях (скорость, память, интерпретируемость)")

# Общие рекомендации по F1-score
all_f1_scores = []
for suture_type, architectures in all_data.items():
    for arch, f1_scores in architectures.items():
        all_f1_scores.extend(f1_scores)

overall_median_f1 = np.median(all_f1_scores)
overall_mean_f1 = np.mean(all_f1_scores)

print(f"\n📈 ОБЩИЕ НАБЛЮДЕНИЯ ПО F1-SCORE:")
print(f"   📊 Общая медиана F1 по всем архитектурам и типам швов: {overall_median_f1:.4f}")
print(f"   📊 Общее среднее F1 по всем архитектурам и типам швов: {overall_mean_f1:.4f}")

if overall_median_f1 >= 0.8:
    performance_level = "отличная"
elif overall_median_f1 >= 0.7:
    performance_level = "хорошая"
elif overall_median_f1 >= 0.6:
    performance_level = "удовлетворительная"
else:
    performance_level = "требует улучшения"

print(f"   🎭 Общий уровень производительности: {performance_level}")

print(f"\n📈 МЕТОДОЛОГИЧЕСКИЕ ВЫВОДЫ:")
print(f"   • F1-score обеспечивает сбалансированную оценку precision и recall")
print(f"   • Тест Краскела-Уоллиса подходит для малых выборок (n=5)")
print(f"   • Непараметрический подход устойчив к нарушениям нормальности")
print(f"   • Поправка Бонферрони консервативна, снижает ложноположительные результаты")
print(f"   • Размер эффекта (η²) помогает оценить практическую значимость различий")
print(f"   • Сравнение медиан более устойчиво к выбросам, чем сравнение средних")



ИТОГОВАЯ СВОДКА СТАТИСТИЧЕСКОГО АНАЛИЗА F1-SCORE
      Тип шва  Архитектуры H-статистика p-value (Kruskal-Wallis) Эффект (η²) Значимость Значимые пары Лучшая архитектура (F1) Худшая архитектура (F1) Размах медиан F1
         COOS            8       4.9452                 0.666656      0.0000        Нет           0/0    DenseNet121 (0.8333)    InceptionV3 (0.7059)           0.1275
          ILS            8       4.7137                 0.694854      0.0000        Нет           0/0          VGG16 (1.0000)    InceptionV3 (0.8571)           0.1429
IOVS_external            8       5.5200                 0.596775      0.0000        Нет           0/0       Xception (0.9583) EfficientNetB0 (0.9020)           0.0564
IOVS_internal            8       9.2835                 0.232938      0.0714        Нет           0/0     ResNet50V2 (0.9268)    InceptionV3 (0.8718)           0.0550

ОБЩИЕ ВЫВОДЫ ПО F1-SCORE
📊 Проанализировано типов швов: 4
🔬 Анализируемая метрика: F1-score (гармоническое среднее